In [1]:
#SI SATURATION RATES (Readme)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Module conducts the following analysis:

# ---- Supplementary Figure 11: Levelized cost of direct electrification, CCUS and biomass technologies

# Module is input for:

# ---- NA

In [2]:
# set working directory and load required packages

setwd("/home/h1604190/Spatially-informed Demand-side Policies for Green H2 Diffusion/") 

# install missing packages and load all dependencies
check_and_load <- function(packages) {
  for (pkg in packages) {
    if (!requireNamespace(pkg, quietly = TRUE)) {
      message(paste("Installing missing package:", pkg))
      install.packages(pkg, dependencies = TRUE, repos = "https://cloud.r-project.org")
    }
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
  }
}

# define required packages for analysis
required_packages <- c(
  "sf",          # spatial data handling
  "dplyr",       # data manipulation
  "readr",       # fast data import
  "readxl",      # read excel files
  "jsonlite",    # read/write json
  "giscoR",      # eurostat/gisco spatial data
  "geosphere",   # distance calculations
  "stringdist",  # fuzzy string matching
  "data.table",  # fast data processing
  "ggplot2",     # plotting
  "ggsci",       # scientific color palettes
  "lubridate",   # date handling
  "tidyr",       # data reshaping
  "stringr",     # string operations
  "patchwork"    # combine plots
)

# load all required packages
check_and_load(required_packages)

# set proj library path for spatial operations
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")

In [3]:
df <- read_xlsx("Data/supplementary_figure_11_data.xlsx")

In [4]:
# theme
plot_theme <- theme_minimal(base_size = 20) +
  theme(
    panel.grid       = element_blank(),
    panel.border     = element_rect(color = "black", fill = NA),
    axis.line        = element_line(color = "black"),
    axis.text        = element_text(size = 20),
    axis.text.y      = element_text(hjust = 1),
    strip.background = element_blank(),
    strip.text       = element_text(size = 20, face = "plain"),
    plot.title       = element_text(size = 20, face = "bold"),
    legend.position  = "bottom",
    legend.title     = element_text(size = 18),
    legend.text      = element_text(size = 16)
  )

# consistent technology colors
tech_colors <- c(
  "Direct Electrification" = "#E41A1C",
  "CCUS"                   = "#000000",
  "Hydrogen"               = "#2C7FB8",
  "Biomass"                = "#31A354"
)

years_keep <- c(2025L, 2030L, 2050L)

df_long <- df %>%
  mutate(
    Process  = str_squish(Process),
    Scenario = str_squish(Scenario),
    type     = str_squish(type)
  ) %>%
  pivot_longer(
    cols = matches("^20\\d{2}$"),
    names_to = "year",
    values_to = "value"
  ) %>%
  mutate(year = as.integer(year))

df_ranges <- df_long %>%
  filter(year %in% years_keep) %>%
  group_by(Process, type, year) %>%
  summarise(
    v_min  = min(value, na.rm = TRUE),
    v_max  = max(value, na.rm = TRUE),
    v_mean = mean(value, na.rm = TRUE),
    .groups = "drop"
  )

# order processes by 2050 mean
proc_order <- df_ranges %>%
  filter(year == 2050L) %>%
  arrange(v_mean) %>%
  pull(Process)

df_ranges <- df_ranges %>%
  mutate(
    Process = factor(Process, levels = proc_order),
    year    = factor(year, levels = c(2025, 2030, 2050))
  )

x_lim <- range(c(df_ranges$v_min, df_ranges$v_max), na.rm = TRUE)

make_panel <- function(df_sub, title, show_y = FALSE) {

  ggplot(df_sub, aes(y = Process, colour = type)) +
    
    geom_errorbar(
      aes(xmin = v_min, xmax = v_max),
      orientation = "y",
      width       = 0,
      linewidth   = 0.9,
      alpha       = 0.35
    ) +
    
    geom_point(
      aes(x = v_mean),
      size = 2.8
    ) +
    
    scale_colour_manual(
      values = tech_colors,
      name   = "Technology"
    ) +
    
    scale_x_continuous(
      limits = x_lim,
      expand = expansion(mult = c(0.01, 0.12))
    ) +
    
    labs(
      title = title,
      x     = NULL,
      y     = NULL
    ) +
    
    plot_theme +
    
    theme(
      axis.text.y  = if (show_y) element_text() else element_blank(),
      axis.ticks.y = if (show_y) element_line() else element_blank()
    )
}

p_2025 <- make_panel(
  df_ranges %>% filter(year == "2025"),
  "2025",
  show_y = TRUE
)

p_2030 <- make_panel(
  df_ranges %>% filter(year == "2030"),
  "2030",
  show_y = FALSE
)

p_2050 <- make_panel(
  df_ranges %>% filter(year == "2050"),
  "2050",
  show_y = FALSE
) +
  labs(x = "EUR MWh-1")

options(repr.plot.width = 18, repr.plot.height = 8, repr.plot.res = 600)

figure_tech_ranges <- (p_2025 | p_2030 | p_2050) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom")

figure_tech_ranges <- figure_tech_ranges +
  plot_annotation(tag_levels = "a")

print(figure_tech_ranges)

ggsave(
  "supplementary-figure-11.pdf",
  figure_tech_ranges,
  device = cairo_pdf,
  width  = 18,
  height = 8,
  dpi    = 600
)